# Using CIM Incrementals

Network models change all the time: equipment is added, removed or modified, and switches open and close. Rather than exchanging the whole model after every change, CIM exchanges a **difference model** (IEC 61970-552) that holds only what changed:

- **forward differences**: the statements the change adds (the new values),
- **reverse differences**: the statements the change removes (the old values).

Every CIM-Graph model (`FeederModel`, `BusBranchModel`, `NodeBreakerModel`, ...) records each `create`, `modify` and `delete` in `network.incrementals`. It can write them as a difference file, and another copy of the model can apply that file.

## The incrementals dictionary

`network.incrementals` has one entry per direction, keyed by class and then by object URI:

```python
{
    'forwardDifferences': {cim_class: {uri: {attribute: value_after}}},
    'reverseDifferences': {cim_class: {uri: {attribute: value_before}}},
}
```

- The forward value is the attribute's value now; the reverse value is its value before the first change. An attribute changed back to its original value has no difference.
- A referenced object is recorded by its URI.
- A created object has `'rdf:type'` (its class name) in its forward difference; a deleted object has it in its reverse difference.
- Only attributes that are written to files are recorded. An association is written from one end (e.g. `Terminal.ConductingEquipment`, not `ConductingEquipment.Terminals`), so it is recorded on that end.

Load the IEEE 13 node feeder:

In [1]:
import os
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
import cimprofiles.cim17.cimhub_2023 as cim

from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel

file = XMLFile(filename='../../tests/test_models/ieee13.xml')
network = FeederModel(container=cim.Feeder(), connection=file, distributed=False)
network.incrementals

{'forwardDifferences': defaultdict(dict, {}),
 'reverseDifferences': defaultdict(dict, {})}

A small helper prints the differences by class name:

In [2]:
def show_incrementals(network):
    for direction, classes in network.incrementals.items():
        print(direction)
        if not classes:
            print('  (none)')
        for cim_class, objects in classes.items():
            for uri, attributes in objects.items():
                print(f'  {cim_class.__name__} {uri}: {attributes}')

## Modifying an object

Change attributes with `network.modify(obj, attribute, value)` rather than by assignment. It checks the value against the attribute's datatype, updates the other end of an association, and records the change:

In [3]:
switch = network.find_by_attribute(cim.LoadBreakSwitch, 'name', 'sect1')[0]
network.modify(switch, 'open', True)
show_incrementals(network)

forwardDifferences
  LoadBreakSwitch 2858B6C2-0886-4269-884C-06FA8B887319: {'open': True}
reverseDifferences
  LoadBreakSwitch 2858B6C2-0886-4269-884C-06FA8B887319: {'open': False}


Changing the switch back to its original value leaves no difference:

In [4]:
network.modify(switch, 'open', False)
show_incrementals(network)

network.modify(switch, 'open', True)  # open it again for the rest of this notebook

forwardDifferences
  (none)
reverseDifferences
  (none)


## Creating objects

`network.create(cim_class, **attributes)` adds a new object to the graph and records it. Here a new line gets a terminal. The line's `Terminals` list is updated too, but only `Terminal.ConductingEquipment` is recorded, because that is the end that is written:

In [5]:
line = network.create(cim.ACLineSegment, name='new_line', length=cim.Length(0.5, 'km'))
terminal = network.create(cim.Terminal, name='new_line_T1', sequenceNumber=1, ConductingEquipment=line)

print('line.Terminals:', [t.name for t in line.Terminals])
show_incrementals(network)

line.Terminals: ['new_line_T1']
forwardDifferences
  LoadBreakSwitch 2858B6C2-0886-4269-884C-06FA8B887319: {'open': True}
  ACLineSegment ad689e99-6f56-56d6-a2c0-2af36bb3c016: {'rdf:type': 'ACLineSegment', 'length': 500.0 meter, 'mRID': 'ad689e99-6f56-56d6-a2c0-2af36bb3c016', 'name': 'new_line'}
  Terminal 0470b028-742d-5dec-8e83-4a99e3fa2f4a: {'rdf:type': 'Terminal', 'ConductingEquipment': 'ad689e99-6f56-56d6-a2c0-2af36bb3c016', 'sequenceNumber': 1, 'mRID': '0470b028-742d-5dec-8e83-4a99e3fa2f4a', 'name': 'new_line_T1'}
reverseDifferences
  LoadBreakSwitch 2858B6C2-0886-4269-884C-06FA8B887319: {'open': False}
  ACLineSegment ad689e99-6f56-56d6-a2c0-2af36bb3c016: {'rdf:type': None, 'length': None, 'mRID': None, 'name': None}
  Terminal 0470b028-742d-5dec-8e83-4a99e3fa2f4a: {'rdf:type': None, 'ConductingEquipment': None, 'sequenceNumber': None, 'mRID': None, 'name': None}


## Deleting objects

`network.delete(obj)` removes an object and every reference to it. Its reverse difference holds all of its values, so the deletion can be undone or checked by the receiver. The references removed from other objects (here the load's terminal and phase) are recorded as changes to those objects. An object that is created and then deleted leaves no difference.

In [6]:
load = network.find_by_attribute(cim.EnergyConsumer, 'name', '611')[0]
network.delete(load)
network.incrementals['reverseDifferences'][cim.EnergyConsumer][load.uri()]

{'customerCount': 1,
 'grounded': True,
 'p': 170000.0 watt,
 'phaseConnection': <PhaseShuntConnectionKind.Y: 'Y'>,
 'q': 80000.0 VAr,
 'LoadResponse': '7E09238A-DF87-4F3A-A192-4FA4C4416A4F',
 'BaseVoltage': '2A158E0C-CD01-4A50-AEBA-59D761FCF15D',
 'EquipmentContainer': '49AD8E07-3BF9-A4E2-CB8F-C3722F837B62',
 'Location': '08866A9B-A318-4DA1-83FE-16EAD7927BA9',
 'mRID': '60B55036-DD71-40CE-ADDA-97B8CE7447DC',
 'name': '611',
 'rdf:type': 'EnergyConsumer'}

## Writing a difference file

`network.write_all_incrementals(filename)` writes every difference recorded so far as an RDF/XML difference model. Created and deleted objects are written as typed elements, changes to existing objects as `rdf:Description`s, and unit values as plain numbers in base SI units. `create`, `modify` and `delete` also take `incremental_file=...`, which writes the same file after the change.

In [7]:
import tempfile

changes_file = os.path.join(tempfile.mkdtemp(), 'changes.xml')
network.write_all_incrementals(changes_file)
print(open(changes_file).read())

<?xml version="1.0" encoding="utf-8"?>
<rdf:RDF xmlns:cim="http://iec.ch/TC57/CIM100#" xmlns:md="http://iec.ch/TC57/61970-552/ModelDescription/1#" xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:dm="http://iec.ch/2002/schema/CIM_difference_model#">
<dm:DifferenceModel rdf:about="urn:uuid:148d3fe9-b1b9-45e5-968f-cf4fc0424f2f">
  <dm:forwardDifferences rdf:parseType="Statements">
    <rdf:Description rdf:about="urn:uuid:2858B6C2-0886-4269-884C-06FA8B887319">
      <cim:Switch.open>true</cim:Switch.open>
    </rdf:Description>
    <cim:ACLineSegment rdf:about="urn:uuid:ad689e99-6f56-56d6-a2c0-2af36bb3c016">
      <cim:Conductor.length>500.0</cim:Conductor.length>
      <cim:IdentifiedObject.mRID>ad689e99-6f56-56d6-a2c0-2af36bb3c016</cim:IdentifiedObject.mRID>
      <cim:IdentifiedObject.name>new_line</cim:IdentifiedObject.name>
    </cim:ACLineSegment>
    <cim:Terminal rdf:about="urn:uuid:0470b028-742d-5dec-8e83-4a99e3fa2f4a">
      <cim:Terminal.ConductingEquipment rdf:res

## Applying a difference file

`network.from_incremental(filename)` applies a difference file to another copy of the model, through `create`, `modify` and `delete`, so the model and its `incrementals` end up as if the changes had been made there. It returns the created and modified objects.

Every difference is checked before any is applied: an unknown class, object or attribute, or a value that does not fit its attribute, raises `ValueError` and the model is not changed. A reverse value that differs from the model's current value (the file was made from another version of the model) is logged as a warning. Only RDF/XML difference files (`.xml`) can be read.

In [8]:
received = FeederModel(container=cim.Feeder(), distributed=False,
                       connection=XMLFile(filename='../../tests/test_models/ieee13.xml'))
changed = received.from_incremental(changes_file)

print('Changed:', [f'{type(obj).__name__} {obj.name}' for obj in changed])
print('Switch open:', received.find_by_attribute(cim.LoadBreakSwitch, 'name', 'sect1')[0].open)
print('Same incrementals:', received.incrementals == network.incrementals)

Changed: ['ACLineSegment new_line', 'Terminal new_line_T1', 'LoadBreakSwitch sect1', 'Terminal 611_T1', 'EnergyConsumerPhase 611_C']
Switch open: True
Same incrementals: True


## Starting a new set of changes

After the differences have been sent, start recording a new set by replacing the dictionary:

In [9]:
from cimgraph.models.incremental_builder import new_differences

network.incrementals = new_differences()
show_incrementals(network)

forwardDifferences
  (none)
reverseDifferences
  (none)


## Summary

- `network.incrementals` records every `create`, `modify` and `delete`: new values in `forwardDifferences`, old values in `reverseDifferences`, by class and object URI.
- Use `network.modify` instead of assigning attributes, so changes are validated and recorded.
- `write_all_incrementals` writes an IEC 61970-552 difference file; `from_incremental` checks and applies one.

The next notebook covers **units**: how CIM-Graph stores and converts physical quantities.